In [ ]:
# —— 排序信号：EP/BM/ROE 优先取月表披露值，缺失时回退到日表月末字段 ——
# EP=EPS/月末收盘价、BM=1/PB 与 features_3 的 get_ep / get_bm 同公式；
# 这里是月表发布值版本，防前视由随后的公告可见性过滤负责。
# 日表月末字段先改名为 *_daily，避免与月表同名字段在合并时冲突。
# 日频和月频信号先分别命名，合并后用月频值，缺失时回退到日频值。
snap = snap.rename(columns={
    eps: eps + '_daily',
    pb: pb + '_daily',
    roe: roe + '_daily',
    public_date: public_date + '_daily',
    end_date: end_date + '_daily',
    close_price: close_price + '_daily',
})
signal_columns = [stock_code, ym, eps, pb, roe, public_date, end_date, monthly_close_price_raw]
source_signal = monthly[signal_columns].copy()
source_signal = source_signal.rename(columns={
    ym: form,
    eps: eps + '_monthly',
    pb: pb + '_monthly',
    roe: roe + '_monthly',
    public_date: public_date + '_monthly',
    end_date: end_date + '_monthly',
    monthly_close_price_raw: 'ClPr_monthly',
})
snap = snap.merge(
    source_signal,
    on=[stock_code, form],
    how='left',
    validate='one_to_one',
)

snap[eps] = snap[eps + '_monthly']
snap.loc[snap[eps].isna(), eps] = snap.loc[snap[eps].isna(), eps + '_daily']
snap[pb] = snap[pb + '_monthly']
snap.loc[snap[pb].isna(), pb] = snap.loc[snap[pb].isna(), pb + '_daily']
snap[roe] = snap[roe + '_monthly']
snap.loc[snap[roe].isna(), roe] = snap.loc[snap[roe].isna(), roe + '_daily']
snap[public_date] = snap[public_date + '_monthly']
snap.loc[snap[public_date].isna(), public_date] = snap.loc[
    snap[public_date].isna(), public_date + '_daily'
]
snap[end_date] = snap[end_date + '_monthly']
snap.loc[snap[end_date].isna(), end_date] = snap.loc[
    snap[end_date].isna(), end_date + '_daily'
]
snap[monthly_close_price_raw] = snap['ClPr_monthly']
snap.loc[snap[monthly_close_price_raw].isna(), monthly_close_price_raw] = snap.loc[
    snap[monthly_close_price_raw].isna(), close_price + '_daily'
]

# 只有公告日不晚于形成月末的财报才能参与排序，避免使用未来信息。
snap[public_date] = pd.to_datetime(snap[public_date], errors='coerce')
visible = snap[public_date] <= snap['formation_end']
snap[monthly_ep] = snap[eps] / snap[monthly_close_price_raw]
snap.loc[~visible, monthly_ep] = np.nan
snap[monthly_bm] = 1 / snap[pb]
snap.loc[~visible | (snap[pb] <= 0), monthly_bm] = np.nan
snap[monthly_roe] = snap[roe] / 100
snap.loc[~visible, monthly_roe] = np.nan



In [ ]:
def prepare_total_asset_events(bs_df):
    required = [bs_stock_code, bs_public_date, bs_end_date, total_ass]
    require_columns(bs_df, required, '资产负债表')

    # 报告期类型不限：AM 取 formation_end 前最近一份已公告报告的总资产。
    events = bs_df[required].dropna(
        subset=[bs_stock_code, bs_public_date, total_ass]
    ).copy()
    # 源行号 → 排序 → (股票, 公告日) 取最后：同键冲突时保留报告期最新的一条。
    events[annual_source_order] = np.arange(len(events))
    events = events.sort_values([
        bs_stock_code, bs_public_date, bs_end_date, annual_source_order,
    ])
    events = events.drop_duplicates(
        [bs_stock_code, bs_public_date], keep='last'
    )
    return events.rename(columns={
        bs_stock_code: annual_stock_code,
        bs_public_date: annual_available_date,
        total_ass: annual_value,
    })




In [ ]:
formation_keys = snap[[stock_code, 'formation_end']].rename(
    columns={'formation_end': date}
)
assets_at_formation = align_annual_value(
    formation_keys, prepare_total_asset_events(bs_df)
)
positive_market_value = snap[monthly_market_value].gt(0)
snap[monthly_am] = (
    assets_at_formation / snap[monthly_market_value]
).where(positive_market_value)

assert not df.duplicated([stock_code, date]).any()
assert not monthly.duplicated([stock_code, ym]).any()
print('清洗与股票池准备完成')
print('df:', df.shape, df[date].min(), df[date].max())
print('monthly:', monthly.shape, monthly[ym].nunique())
print('snap:', snap.shape, int(snap['eligible'].sum()))